# 03 — YOLOv8 Architecture and Loss Formulation

This notebook explores the architectural advancements of Ultralytics YOLOv8 and derives its loss functions: Complete IoU (CIoU) Loss and Distribution Focal Loss (DFL).

---

## 1. Architectural Evolution: C2f, Decoupled Head, and Anchor-Free Design

YOLOv8 introduces three foundational paradigm shifts over prior iterations:

```text
+-------------------------------------------------------------------------------+
|                            YOLOv8 SYSTEM TOPOLOGY                             |
+-------------------------------------------------------------------------------+
| Input Image (3, 640, 640)                                                     |
|       │                                                                       |
|   [Backbone]   Conv, C2f modules, SPPF (Spatial Pyramid Pooling Fast)         |
|       │        Multi-scale feature pyramid: P3 (80x80), P4 (40x40), P5 (20x20)|
|       ▼                                                                       |
|     [Neck]     PANet (Path Aggregation Network: Top-Down & Bottom-Up)         |
|       │                                                                       |
|       ▼                                                                       |
|    [Head]      Anchor-Free Decoupled Head                                     |
|                ├── Classification Branch: BCE / VFL -> Class Probabilities    |
|                └── Regression Branch: CIoU + DFL -> Continuous Bounding Boxes |
+-------------------------------------------------------------------------------+
```

1. **C2f Module**: Replaces the C3 module used in YOLOv5. Combines high-level feature abstraction with cross-stage partial connections, allowing gradient information to flow across multiple parallel branches without bottlenecking.
2. **Anchor-Free Paradigm**: Eliminates manual anchor clustering (k-means on training sets). Instead of predicting offsets from predefined anchor templates, the network directly predicts the distance from each feature grid point to the four bounding box edges $(l, t, r, b)$.
3. **Decoupled Detection Head**: Separates classification and spatial regression tasks into two independent sub-networks. This mitigates conflicting gradient updates between semantic category learning and spatial boundary refinement.

In [ ]:
import math

import numpy as np


def compute_c2f_complexity_summary() -> dict[str, str]:
    """Summarize structural differences between C3 and C2f building blocks."""
    return {
        "C3_module": "1x1 conv + 3x3 bottleneck stack + concatenation (standard CSPNet)",
        "C2f_module": "1x1 conv + multiple parallel split branches + dense skip connections + 1x1 conv",
        "Advantage": "Rich gradient flow, richer feature representation, lower parameter count",
    }


print("Structural Comparison:")
for k, v in compute_c2f_complexity_summary().items():
    print(f"  {k}: {v}")

## 2. Complete IoU (CIoU) Loss Formulation

Standard IoU provides zero gradient when two bounding boxes do not overlap ($\text{IoU} = 0$). While Generalized IoU (GIoU) addresses this using the minimum enclosing box, and Distance IoU (DIoU) minimizes center point distance, **Complete IoU (CIoU)** incorporates three geometric factors:

1. **Overlap Area**: $1 - \text{IoU}$
2. **Normalized Center Distance**: $\frac{\rho^2(b, b^{gt})}{c^2}$
3. **Aspect Ratio Consistency**: $\alpha v$

The mathematical formulation:

$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(b, b^{gt})}{c^2} + \alpha v$$

where:
- $\rho(b, b^{gt})$ is the Euclidean distance between the centroids of $b$ and $b^{gt}$.
- $c$ is the diagonal length of the smallest enclosing box covering both $b$ and $b^{gt}$.
- $v$ quantifies the discrepancy of aspect ratio:
  $$v = \frac{4}{\pi^2} \left( \arctan \frac{w^{gt}}{h^{gt}} - \arctan \frac{w}{h} \right)^2$$
- $\alpha$ is a dynamic balancing parameter that prioritizes overlap when IoU is low:
  $$\alpha = \frac{v}{(1 - \text{IoU}) + v}$$

In [ ]:
def compute_ciou_loss(box_pred: np.ndarray, box_target: np.ndarray) -> tuple[float, float]:
    """Compute Complete IoU (CIoU) Loss between prediction and ground truth in VOC format [xmin, ymin, xmax, ymax].

    Returns:
        (ciou_loss, iou)
    """
    # Coordinates
    x1_p, y1_p, x2_p, y2_p = box_pred
    x1_t, y1_t, x2_t, y2_t = box_target

    w_p = max(0.0, x2_p - x1_p)
    h_p = max(0.0, y2_p - y1_p)
    w_t = max(0.0, x2_t - x1_t)
    h_t = max(0.0, y2_t - y1_t)

    # Intersection
    x1_inter = max(x1_p, x1_t)
    y1_inter = max(y1_p, y1_t)
    x2_inter = min(x2_p, x2_t)
    y2_inter = min(y2_p, y2_t)
    intersection = max(0.0, x2_inter - x1_inter) * max(0.0, y2_inter - y1_inter)

    # Union
    union = (w_p * h_p) + (w_t * h_t) - intersection
    iou = intersection / union if union > 0 else 0.0

    # Smallest enclosing box
    x1_c = min(x1_p, x1_t)
    y1_c = min(y1_p, y1_t)
    x2_c = max(x2_p, x2_t)
    y2_c = max(y2_p, y2_t)
    c2 = (x2_c - x1_c) ** 2 + (y2_c - y1_c) ** 2 + 1e-7

    # Center distance
    center_p = np.array([(x1_p + x2_p) / 2.0, (y1_p + y2_p) / 2.0])
    center_t = np.array([(x1_t + x2_t) / 2.0, (y1_t + y2_t) / 2.0])
    rho2 = np.sum((center_p - center_t) ** 2)

    # Aspect ratio penalty
    arctan_diff = math.atan(w_t / (h_t + 1e-7)) - math.atan(w_p / (h_p + 1e-7))
    v = (4.0 / (math.pi**2)) * (arctan_diff**2)
    alpha = v / (1.0 - iou + v + 1e-7)

    ciou = iou - (rho2 / c2) - (alpha * v)
    ciou_loss = 1.0 - ciou
    return float(ciou_loss), float(iou)


# Benchmark identical, slightly displaced, and distant boxes
gt = np.array([100.0, 100.0, 200.0, 300.0])
pred_perfect = np.array([100.0, 100.0, 200.0, 300.0])
pred_offset = np.array([110.0, 105.0, 210.0, 305.0])
pred_aspect_distorted = np.array([100.0, 100.0, 300.0, 200.0])

l_perf, iou_perf = compute_ciou_loss(pred_perfect, gt)
l_off, iou_off = compute_ciou_loss(pred_offset, gt)
l_asp, iou_asp = compute_ciou_loss(pred_aspect_distorted, gt)

print(f"Perfect match:       IoU={iou_perf:.3f}, CIoU Loss={l_perf:.4f}")
print(f"Offset match:        IoU={iou_off:.3f}, CIoU Loss={l_off:.4f}")
print(f"Aspect distorted:    IoU={iou_asp:.3f}, CIoU Loss={l_asp:.4f}")

assert l_perf < 1e-5
assert l_off < l_asp

## 3. Distribution Focal Loss (DFL)

In real images, object boundaries are often ambiguous due to occlusions, shadows, or motion blur. Treating bounding box coordinates as sharp Dirac delta distributions $\delta(x - y)$ can lead to brittle regression.

**Distribution Focal Loss (DFL)** models continuous coordinate distances as general probability distributions over discrete bins $\{y_0, y_1, \dots, y_n\}$.

Given target continuous coordinate $y$ bounded by discrete integer bins $y_i \le y \le y_{i+1}$, the network predicts probability vector $S = \text{softmax}(z)$. The DFL forces probability mass to concentrate on the two bins nearest to $y$:

$$\mathcal{L}_{\text{DFL}}(S_i, S_{i+1}) = - \left( (y_{i+1} - y) \log S_i + (y - y_i) \log S_{i+1} \right)$$

The continuous prediction is recovered as the expectation:
$$\hat{y} = \sum_{j=0}^n S_j \cdot y_j$$

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    e = np.exp(logits - np.max(logits))
    return e / np.sum(e)


def compute_dfl_loss(logits: np.ndarray, target: float) -> tuple[float, float]:
    """Compute Distribution Focal Loss for a single coordinate.

    Args:
        logits: (n_bins,) raw network outputs for coordinate distribution
        target: Float ground-truth continuous offset
    Returns:
        (dfl_loss, expected_coordinate)
    """
    probs = softmax(logits)
    bins = np.arange(len(logits), dtype=float)
    expected_val = float(np.sum(probs * bins))

    y_i = int(np.clip(np.floor(target), 0, len(logits) - 2))
    y_i1 = y_i + 1

    weight_left = y_i1 - target
    weight_right = target - y_i

    eps = 1e-8
    loss = -(weight_left * np.log(probs[y_i] + eps) + weight_right * np.log(probs[y_i1] + eps))
    return float(loss), expected_val


# Test DFL with 16 bins for target coordinate y = 6.4
n_bins = 16
target_coord = 6.4

# Logits concentrated around bin 6 and 7
optimal_logits = np.zeros(n_bins)
optimal_logits[6] = 2.5
optimal_logits[7] = 3.0

loss_val, predicted_coord = compute_dfl_loss(optimal_logits, target_coord)
print(f"Target coordinate:    {target_coord}")
print(f"Predicted coordinate: {predicted_coord:.3f}")
print(f"DFL loss:             {loss_val:.4f}")
assert abs(predicted_coord - target_coord) < 0.5